# CasDsl — a categorically organized CAS in Lean 4

Every cell in this notebook is written in CasDsl, a computer-algebra
language [elaborated](https://lean-lang.org/doc/reference/latest/Elaboration-and-Compilation/)
by Lean 4 — type-checked by the same machinery that verifies proofs,
though no cell creates a theorem; an assertion is only as strong as the
backend that answered it. You need no Lean to read the cells; the
notation is ordinary mathematics.

**The organizing philosophy: everything is a category.** A value is
instantiated into its categories, and the category decides which methods
it carries: `2 + 2i` is a complex number, so it has `.re()`, `.im()`,
`|·|`; a square matrix has `.det()`; a polynomial has `.roots()` and
`.factor()`. This is what makes Sage a "catalogue of algorithms" — with
the categories actually defined, and methods that travel along
inclusions by functorial transport: ℤ ≤ ℚ ≤ ℝ ≤ ℂ, Euclidean ≤ UFD, so
an operation is declared once, where it first makes sense, and inherited
everywhere below.

**Backend-blind syntax.** No expression ever names Sage, GAP, or an
algorithm: you ask for the mathematical operation, and a backend answers
— Sage today, with Python, GAP, Singular, Julia, and native binaries in
the pool. Backends are interchangeable, and the notation never names them.

In an ordinary Jupyter notebook, the kernel is a REPL: context persists
between cell runs, and every run — in any order, any number of times —
appends to that context. A cell's output records the context at the
moment it ran, which can diverge from the notebook as it reads on
screen: re-run a cell or run cells out of order, and outputs no longer
line up with the code. This notebook runs on a kernel that excludes that
failure: the notebook is a **document, not a transcript**. Each cell's
output is the result of elaborating the visible prefix of the notebook
through that cell, so outputs always correspond to the code shown above
them. Editing a cell re-runs it and re-establishes everything below it —
visibly, marked with `↻`, never silently. Unchanged cells are cached, so
editing cell 5 does not re-run cell 1; an upstream failure stops the run
with an `UpstreamError` rather than leaving the notebook half-applied.
In an ordinary notebook, editing is free and outputs drift from the
code; here editing makes the notebook recompute — surprising until you
see the point: what you read is what was computed.


## 1 · Assertions

`assert` takes a **proposition** — a decidable statement. The primitive
proposition is equality: `2 + 3 = 5` claims the two sides denote the
same element of the ambient domain. The other relations — `≠`, `∈`,
`∉`, `⊆` — form propositions too, and `and` conjoins several into one
assertion.

A proposition is three-valued: `true`, `false`, or `unknown`. `assert P`
demands `true` — that commits the cell. `false` fails it, rolling the
notebook state back to the last committed cell. `unknown` can occur when
the two sides could not be compared in the ambient domain; it is a
failure too, distinct from `false`. A fourth outcome, `error`, is not a
truth value but a machine failure: no implementation exists or a backend
died, and the system says so.

Nothing is proved: no Lean theorem is generated. `assert 2 + 3 = 5`
holds because a backend computed `2 + 3` and got `5` — the assertion's
whole content is what the backend reported.


### Reserved notation

The grammar's reserved words and symbols, with what they do:

| token | meaning |
|---|---|
| `let x := e in D` | bind `x` to `e`, ascribed to a domain or category `D` (checked) |
| `let p(x) := e in D` | define `p` with indeterminate `x` — e.g. `ℤ[x]` |
| `assert P` | decide `P` — three truth values, §1 |
| `map e to D` | transport `e` along the canonical map into `D` |
| `x ↦ e` | a function; `|->` is the ASCII spelling |
| `→`, `->` | the domain arrow of a function type |
| `=`, `is`, `≠`, `∈`, `in`, `∉`, `⊆` | assertion relations (`\in`, `\leq` are the backslash spellings) |
| `√` | a chosen branch of the multivalued square root: `√2` is the non-negative root, `√−1` is the primitive fourth root of unity, `i` |
| `i`, `e` | never shadowable: the imaginary unit (a primitive fourth root of unity), Euler's number (`e^t = exp t`) |
| `d`, `π` | shadowable constants: the differential, pi |
| `dx` | the differential atom — `∫ f dx`, `d(f) = (6x + 1) dx` |
| `O(ε)` | approximation tolerance: `map √2 to ℝ/O(1/10^{10})` |
| `…` | set continuation: `{0, 2, 4, …}` |
| `#capabilities` | inspect the dispatch: also `#explain_route e`, `#capability_gaps` |

Only the first column's true tokens — `dx`, `Spec`, `lim_`, `map`, `to`,
`is`, and the constants `i`, `e` — are words no binding may shadow; the
other spellings (`and`, `O`, `dt`) are ordinary identifiers elsewhere in
the grammar.


In [1]:
assert 2 + 3 = 5

Starting Lean worker (/home/dzack/gitclones/lean-cas-dsl)…


✓ 2 + 3 = 5


The `in ℤ/5` suffix changes the ring the assertion is evaluated in.
`2 + 3` is 5 in ℤ, but 5 ≡ 0 in ℤ/5, so the assertion holds.

In [2]:
assert 2 + 3 = 0 in ℤ/5

✓ 2 + 3 = 0 in ℤ/5


`and` chains decide conjunct by conjunct: one commit when every conjunct is true, and a refusal names the conjunct that failed. `is` is just `=` in the SPEC's spelling.

In [3]:
assert 2 + 3 = 5 and 4 ∉ {1, 2, 3}

✓ 2 + 3 = 5 and 4 ∉ {1, 2, 3}


In [4]:
assert 2 + 3 is 5

✓ 2 + 3 = 5


The reserved notation bites in bindings too: `e` is Euler's constant and `i` the imaginary unit, so no binding may shadow either — both mean the same thing in every session state. The refusals below are the point: they fail loudly instead of silently rebinding.

In [5]:
let e := 5 in ℤ

LeanError: `e` is reserved for Euler's constant (`e^t` = exp(t)) and cannot be rebound

In [6]:
let i := 3 in ℤ

LeanError: `i` is reserved for the imaginary unit i ∈ ℂ, so it cannot be rebound

## 2 · Factorization

`factor` is a method on elements of a unique factorization domain (UFD):
the category is named for the theorem that holds there — factorization
into irreducibles, unique up to units and order. An integer receives it
because ℤ is Euclidean and every Euclidean domain is a UFD: the method
arrives along the inclusion, not by per-ring forwarding code. Sage
performs the computation, but `n.factor()` never names it.


In [7]:
let n := 360 in ℤ

n := 360 ∈ ℤ

In [8]:
n.factor()

Error in sitecustomize; set PYTHONVERBOSE for traceback:


NameError: name 'TYPE_CHECKING' is not defined


2^3 * 3^2 * 5

Routes are inspectable: `#explain_route` names the backend and the route a computation took, textually. And a literal numeral is not a receiver — `360.factor()` is a tokenizer casualty (the lexer reads `360.` as a decimal before any production sees it) — so the receiver is parenthesized, and the method resolves on the receiver's VALUE.

In [9]:
#explain_route n.factor()

factor (x : R) [CommRing R] [IsDomain R] [UniqueFactorizationMonoid R]
  ≐ UniqueFactorizationMonoid.factors — stated up to units; answers are normalized — positive leading unit in ℤ, monic factors over a field
x = 360 ∈ ℤ; R = ℤ : EuclideanDomain ≤ IsPrincipalIdealRing ≤ UniqueFactorizationMonoid  (synthesized)
route: sage "factor_int" — implemented for element of ℤ
source: https://github.com/dzackgarza/lean-cas-dsl/blob/main/backends/sage_adapter.py
result: a factorization: a unit and irreducible factors with multiplicity

In [10]:
(360).factor()

2^3 * 3^2 * 5

In [11]:
assert (84).gcd(30) = 6

✓ (84).gcd(30) = 6


`gcd` works the same way — a method on UFD elements, where a greatest
common divisor is unique up to units, and ℤ inherits it through the same
inclusion.


In [12]:
gcd(84, 30) = 6

true

## 3 · Polynomials

`roots()` and `factor()` are answered in the ring a polynomial is bound
to: the same $p$ can live in $\mathbb{Z}[x]$, $\mathbb{Q}[x]$, or
$\mathbb{C}[x]$, and the answer changes with the ring.

The `(x)` in `let p(x) := … in ℤ[x]` declares `x` as the indeterminate.

In [13]:
let p(x) := x^5 - x^4 - x^3 + x^2 - 2x + 2 in ℤ[x]

p := x^5 - x^4 - x^3 + x^2 - 2x + 2 ∈ ℤ[x]

Evaluation is substitution. Is $1$ a root?

In [14]:
assert p(1) = 0

✓ p(1) = 0


`p.roots()` asks for the roots **in the coefficient ring**, as a
**multiset** — the anchor `Polynomial.roots` is multiset-valued, so a
root carries its multiplicity and `|·|` counts with it. How many
roots does $p$ have in ℤ?

In [15]:
p.roots()

p does not split over ℤ: it has 1 of its 5 roots there (with multiplicity); `map p to ℂ[x]` reaches the rest


{1}

`p.factor()` asks the companion question: the factorization into
irreducibles, with multiplicity. The irreducible quadratic factors are
precisely the roots the ring cannot see.

In [16]:
p.factor()

(x - 1) * (x^2 - 2) * (x^2 + 1)

Move $p$ to $\mathbb{Q}[x]$ along the canonical inclusion
$\mathbb{Z} \subseteq \mathbb{Q}$, coefficient by coefficient.

For a monic integer polynomial, Gauss: rational roots are integers, and
irreducibility over ℤ ⟺ over ℚ. ℚ is not a splitting field for either
quadratic, so the quadratics stay irreducible.


In [17]:
let pQ := map p to ℚ[x]

pQ := x^5 - x^4 - x^3 + x^2 - 2x + 2 ∈ ℚ[x]

In [18]:
pQ.roots()

pQ does not split over ℚ: it has 1 of its 5 roots there (with multiplicity); `map pQ to ℂ[x]` reaches the rest


{1}

In [19]:
pQ.factor()

(x - 1) * (x^2 - 2) * (x^2 + 1)

Multiplicity is part of the answer. $(x-1)^2$ has one rational root,
counted **twice**: the multiset shows both copies and `|·|` counts 2.
The set literal `{1}` asserts a *simple* root — for this polynomial
that claim is honestly false.

In [20]:
let dbl(x) := x^2 - 2x + 1 in ℚ[x]

dbl := x^2 - 2x + 1 ∈ ℚ[x]

In [21]:
dbl.roots()

{1, 1}

In [22]:
assert |dbl.roots()| = 2

✓ |dbl.roots()| = 2


Map to $\mathbb{C}[x]$. ℂ is algebraically closed: $p$ splits into
linear factors, and the root multiset becomes the full census. The census is
taken in the ring asked for, and no further.


In [23]:
let pC := map pQ to ℂ[x]

pC := x^5 - x^4 - x^3 + x^2 - 2x + 2 ∈ ℂ[x]

In [24]:
pC.roots()

algebraic numbers are presented through a fixed embedding ℚ̄ ↪ ℂ


{-√2, 1, √2, -i, i}

In [25]:
pC.factor()

algebraic numbers are presented through a fixed embedding ℚ̄ ↪ ℂ


(x - √2) * (x - 1) * (x - i) * (x + i) * (x + √2)

Definitions use `:=` — SPEC's opening sentence. `NAME := expr` is a command, sugar for `let` through the same elaborator; SPEC's own §Polynomials line is `q := map p to ℂ[x]`. Maps along canonical inclusions compose freely with the polynomial commands.

In [26]:
let pb(x) := x^3 - 2x + 1 in ℤ[x]

pb := x^3 - 2x + 1 ∈ ℤ[x]

In [27]:
qb := map pb to ℂ[x]

qb := x^3 - 2x + 1 ∈ ℂ[x]

In [28]:
assert qb(1) = 0

✓ qb(1) = 0


The roots by type: $1$ (the integer root); $\pm\sqrt{2}$ — a pair
that travels together, the two roots of the irreducible $x^2 - 2$; and
$\pm i$ — the complex-conjugate pair. Three irreducibles over ℤ become
five linear factors over ℂ. Nothing about $p$ changed — the ring did.

`|·|` counts the multiset with multiplicity — five simple roots
here. The Σ/Π reads — elementary symmetric functions of the roots —
ride the solution-set spelling `{a ∈ ℂ | pC(a) = 0}`, the same
spelling SPEC's composed computation uses.

In [29]:
assert |pC.roots()| = 5
let census := {a ∈ ℂ | pC(a) = 0} in 𝒫(ℂ)
assert ∑_{a ∈ census} a = 1
assert ∏_{a ∈ census} a = -2

algebraic numbers are presented through a fixed embedding ℚ̄ ↪ ℂ


✓ |pC.roots()| = 5


algebraic numbers are presented through a fixed embedding ℚ̄ ↪ ℂ


✓ ∑_{a ∈ census} a = 1


✓ ∏_{a ∈ census} a = -2


census := {-√2, 1, √2, -i, i}

## 4 · Exact algebraic numbers

CasDsl works with exact algebraic numbers — never decimals unless you
explicitly request an approximation. The ⊆-chain
$\mathbb{N} \subseteq \mathbb{Z} \subseteq \mathbb{Q} \subseteq \mathbb{R} \subseteq \mathbb{C}$
is built in: membership and transport are automatic.


In [30]:
let z := 2 + 2i in ℂ

z := 2 + 2i ∈ ℂ

The absolute value is an exact algebraic operation: `|·|` returns the
surd — simplification carried out exactly, never approximated as a
decimal.

In [31]:
assert |z| = 2√2

√ denotes the principal branch: the non-negative root for a positive radicand, i·√|d| for a negative one


✓ |z| = 2√2


Numerical approximation is an operation **on** an exact value, not a
replacement for it. `map √2 to ℝ/O(1/10^{10})` asks for $\sqrt{2}$
approximated to within $10^{-10}$ in ℝ. The tolerance is a request, not a
quotient — the underlying value remains exact.

In [32]:
map √2 to ℝ/O(1/10^{10})

√ denotes the principal branch: the non-negative root for a positive radicand, i·√|d| for a negative one


1.4142135623 + O(1/10^{10})

Membership is a decided proposition, and the exact values live where they belong: `√2 ∈ ℝ` and `2 + 2i ∈ ℂ` are SPEC's own lines — no decimal anywhere.

In [33]:
assert √2 ∈ ℝ

√ denotes the principal branch: the non-negative root for a positive radicand, i·√|d| for a negative one


✓ √2 ∈ ℝ


In [34]:
assert 2 + 2i ∈ ℂ

✓ 2 + 2i ∈ ℂ


The number-system chain is decided by the registry of canonical maps: `ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ` is SPEC's own line, one commit. The visible consequence: an integer ascribes to ℝ or ℂ through a registered canonical map.

In [35]:
assert ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ

✓ ℤ ⊆ ℚ and ℚ ⊆ ℝ and ℝ ⊆ ℂ


In [36]:
let rr := 3 in ℝ

rr := 3 ∈ ℝ

In [37]:
map 3 to ℂ

3

The bare-definition sugar carries its ascription tail: `zb := 2 + 2i in ℂ` checks the membership exactly as `let` would.

In [38]:
zb := 2 + 2i in ℂ

zb := 2 + 2i ∈ ℂ

In [39]:
assert zb.re() = 2

✓ zb.re() = 2


## 5 · Linear algebra

Exact matrix arithmetic over ℚ. Matrix literals use **row-semicolon
syntax**: `[1, 2; 3, 4]` is a $2\times 2$ matrix whose rows are `[1, 2]`
and `[3, 4]`.

In [40]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

M := [1, 2; 3, 4] ∈ Mat₂(ℚ)

The determinant is a category-owned method on square matrices, exact
over ℚ — no floating point.

In [41]:
assert M.det() = -2

✓ M.det() = -2


`M⁻¹` exists because $\det(M) \ne 0$, and over ℚ it is computed
exactly: rational entries, no decimals.

In [42]:
M⁻¹

[-2, 1; 3/2, -1/2]

Rank and kernel are category-owned methods: `M` is invertible, so it kills nothing — `ker M = {0}`, and `0` is the zero of the ambient space ℚ².

In [43]:
assert M.rank() = 2

✓ M.rank() = 2


In [44]:
assert M.ker() = {0}

✓ M.ker() = {0}


In [45]:
M.ker()

{0} ≤ ℚ²

`and` survives identifier conjuncts — every conjunct here ends in a NAME, the shape the juxtaposition parser is careful about.

In [46]:
let v := (1, 0) in ℚ²

v := (1, 0) ∈ ℚ²

In [47]:
let b := (1, 3) in ℚ²

b := (1, 3) ∈ ℚ²

In [48]:
assert M v = b and v = v

✓ M v = b and v = v


Transport along a forgetful functor: `F` is the ℤ-module ℤ/4 in `SmallModules(ℤ)`, a subcategory of `Modules(ℤ)`, which is a subcategory of `Sets`. Module methods resolve directly — the annihilator needs no help — while cardinality is a Sets question that arrives through the functor, and `#explain_route` shows the path.

In [49]:
let F := ℤ/4 in SmallModules(ℤ)

F := ℤ/4 as ℤ-module

In [50]:
F.annihilator()

(4)

In [51]:
F.cardinality()

4

In [52]:
#explain_route F.cardinality()

cardinality (x : R)
  ≐ Cardinal.mk
x = ℤ/4 as ℤ-module : Finite ≤ Countable ≤ Sets  (synthesized)
via UnderlyingSet : Modules ⟶ Sets, image {0, 1, 2, 3}
route: native "cardinality" — implemented for any set
source: https://github.com/dzackgarza/lean-cas-dsl/blob/main/CasDsl/Native.lean
result: a cardinal (finite n, or ℵ₀)

Bare `=` never inserts the functor: `F` is a module, `{0, 1, 2, 3}` is a set, and cross-category equality is trivially false — the refusal says so. The Sets question is one explicit call away, receiver transported:

In [53]:
assert F = {0, 1, 2, 3}

LeanError: assertion is false: F = {0, 1, 2, 3}

In [54]:
F.set_eq({0, 1, 2, 3})

true

## 6 · Calculus

CasDsl distinguishes the **universal differential** $d(f)$ — a 1-form —
from the **derivation** $(d/dx)(f)$ — a polynomial. They are different
types and not equal, even when their coefficients match.

The indefinite integral $\int f\,dx$ returns the **coset** of
antiderivatives — the set of all primitives, not a choice of constant.

In [55]:
let f := x ↦ 3x² + x + 1 in ℚ[x]

f := 3x^2 + x + 1 ∈ ℚ[x]

`d` is the universal relative differential: `d(f)` is a 1-form, and the
`dx` is part of the value, not decoration.

In [56]:
assert d(f) = (6x + 1) dx

✓ d(f) = (6x + 1) dx


`(d/dx)(f)` is the derivation: a plain polynomial. Note the absence of
`dx` — this is the coefficient of the differential, not the
differential itself.

In [57]:
assert (d/dx)(f) = 6x + 1

✓ (d/dx)(f) = 6x + 1


The $+ \mathbb{Q}$ is the constant of integration, presented as a coset:
any rational constant can be added to a primitive and the result is
still a primitive. This is not a notational convention — the integral is
a set.


In [58]:
∫ f dx

x^3 + (1/2)x^2 + x + ℚ

The kernel of the derivation is exactly the constant field:

In [59]:
assert kernel(d/dx : ℚ[x] → ℚ[x]) = ℚ

✓ kernel(d/dx : ℚ[x] → ℚ[x]) = ℚ


The coset is usable, not just displayable: select the primitives with
`h(0) = 0` — there is exactly one — pick it out, and check it against
the differential and the derivation.

In [60]:
let Fs := {h ∈ ∫ f dx | h(0) = 0} in 𝒫(ℚ[x])
assert Fs.cardinality() = 1
let F := Fs[0] in ℚ[x]
assert F(x) = x³ + (1/2)x² + x
assert d(F) = f dx

✓ Fs.cardinality() = 1


✓ F(x) = x³ + (1/2)x² + x


✓ d(F) = f dx


Fs := {x^3 + (1/2)x^2 + x}

F := x^3 + (1/2)x^2 + x ∈ ℚ[x]

## 7 · Finite sets

Sets are values, ascribable to a power-set category: `𝒫(ℤ)` and `2^ℤ`
are the same collection read two ways. The four binary operations —
union, intersection, difference, symmetric difference — are exact.

In [61]:
let A := {1, 2, 3} in 𝒫(ℤ)
let B := {3, 4, 5} in 2^ℤ

A := {1, 2, 3}

B := {3, 4, 5}

In [62]:
assert A ∪ B = {1, 2, 3, 4, 5}
assert A ∩ B = {3}
assert A \ B = {1, 2}
assert A △ B = {1, 2, 4, 5}

✓ A ∪ B = {1, 2, 3, 4, 5}


✓ A ∩ B = {3}


✓ A \ B = {1, 2}


✓ A △ B = {1, 2, 4, 5}


In [63]:
assert |A| = 3
assert |A × B| = 9
assert |𝒫(A)| = 2^|A|

✓ |A| = 3


✓ |A × B| = 9


✓ |𝒫(A)| = 2^|A|


Membership and inclusion are propositions: `∈`, `∉`, `⊆`. The
power-set ascription is **checked** — a set containing a non-integer
element would be refused in `𝒫(ℤ)`.

In [64]:
assert 2 ∈ A
assert 4 ∉ A
assert A ⊆ A ∪ B
assert A ∩ B ⊆ A

✓ 2 ∈ A


✓ 4 ∉ A


✓ A ⊆ A ∪ B


✓ A ∩ B ⊆ A


Comprehension: `{n ∈ ℤ | n² ≤ 20}` is the subset of ℤ decided by the
guard. An infinite comprehension like `{2n | n ∈ ℕ}` is not a crash —
it is its progression, and membership is decidable.

In [65]:
let S := {n ∈ ℤ | n² ≤ 20}
assert S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}
assert |S| = 9

✓ S = {-4, -3, -2, -1, 0, 1, 2, 3, 4}


✓ |S| = 9


S := {-4, -3, -2, -1, 0, 1, 2, 3, 4}

In [66]:
let m2: ℕ → ℕ := n ↦ 2n
let E := {2n | n ∈ ℕ}
assert 8 ∈ E
assert |E| = ℵ₀

✓ 8 ∈ E


✓ |E| = ℵ₀


m2 := n ↦ 2n ∈ ℕ → ℕ

E := {0, 2, ...}

In [67]:
assert m2(ℕ) = E
assert m2.image() = E

✓ m2(ℕ) = E


✓ m2.image() = E


A bounded image comprehension enumerates exactly — the set is the image
of the finite range, no more, no less:

In [68]:
{m2(n) | n ∈ ℕ, 0 ≤ n < 6}

{0, 2, 4, 6, 8, 10}

## 8 · Functions

A function is an expression, not a table: `t ↦ t² + 1` denotes the map
itself, and two functions are equal when they are the same expression —
equality is not sampled at points.

In [69]:
let h := t ↦ t² + 1 in ℝ → ℝ
let hp(t) := t^2 + 1 in R->R
assert h = hp

✓ h = hp


h := t ↦ t^2 + 1 ∈ ℝ → ℝ

hp := t ↦ t^2 + 1 ∈ ℝ → ℝ

In [70]:
assert h(0) = 1
assert h(3) = 10

✓ h(0) = 1


✓ h(3) = 10


Parity is an identity of expressions, seen without any point check:

In [71]:
assert h(-t) = h(t)

✓ h(-t) = h(t)


Composition is an identity of function expressions too:

In [72]:
let sq(t) = t^2 in RR->RR
let cub(t) = t^3 in RR->RR
assert (sq ∘ cub)(t) = t^6

✓ (sq ∘ cub)(t) = t^6


sq := t ↦ t^2 ∈ ℝ → ℝ

cub := t ↦ t^3 ∈ ℝ → ℝ

A body the polynomial engine cannot express is SYMBOLIC: a presentation, not a point-evaluable function. Nothing here approximates — the refusal says so. `e` is Euler's constant in these bodies, and `sin`, `1/t` are vocabulary the polynomial reading cannot reach.

In [73]:
let expo := t ↦ e^t in ℝ → ℝ

expo := t ↦ e^t ∈ ℝ → ℝ

In [74]:
let sine: ℝ → ℝ := t ↦ sin(t)

sine := t ↦ sin(t) ∈ ℝ → ℝ

In [75]:
let recip := t ↦ 1/t in ℝ → ℝ

recip := t ↦ 1/t ∈ ℝ → ℝ

In [76]:
sine(0)

LeanError: t ↦ sin(t) has a symbolic body: the expression is presented for a limit, a definite integral or a Taylor expansion, and evaluation at a point is not implemented for symbolic bodies

In [77]:
recip(2)

LeanError: t ↦ 1/t has a symbolic body: the expression is presented for a limit, a definite integral or a Taylor expansion, and evaluation at a point is not implemented for symbolic bodies

A lambda needs its domain: the ascription is not optional.

In [78]:
let bad := t ↦ t^2

LeanError: `t ↦ …` needs an ascription naming the domains it runs between, as in `let h := t ↦ t^2 + 1 in ℝ → ℝ`

Composition is refused when the domains do not compose — `h : ℝ → ℝ` after `m2 : ℕ → ℕ` has no common meeting set — and an argument outside the source domain refuses by name:

In [79]:
assert (h ∘ m2)(t) = t

LeanError: ℕ → ℕ and ℝ → ℝ do not compose: the target of the right factor is not the source of the left

In [80]:
m2(-1)

LeanError: -1 is not an element of ℕ

## 9 · Subspaces and spans

`span_QQ{u₁, u₂}` is the ℚ-span of two vectors, ascribed as a
**subobject**: `\leq ℚ³ in Mod(QQ)` — the `≤` is subobject-in-a-
category, not the numeric relation. The span answers dimension and
membership.

In [81]:
let u₁ := (1, 0, 1) in ℚ³
let u₂ := (0, 1, 1) in ℚ³
let W := span_QQ{u₁, u₂} \leq ℚ³ in Mod(QQ)

u₁ := (1, 0, 1) ∈ ℚ³

u₂ := (0, 1, 1) ∈ ℚ³

W := span_ℚ{(1, 0, 1), (0, 1, 1)} ≤ ℚ³

In [82]:
assert W.dim() = 2
assert (1, 1, 2) ∈ W
assert (1, 1, 0) ∉ W

✓ W.dim() = 2


✓ (1, 1, 2) ∈ W


✓ (1, 1, 0) ∉ W


The same subspace read as a kernel: φ vanishes exactly on `W`. A linear
map is a hom in the category, and its kernel is a subobject again.

In [83]:
let φ: ℚ³ → ℚ := (a, b, c) ↦ a + b - c
assert W = ker φ

✓ W = ker φ


φ := (a, b, c) ↦ a + b - c ∈ ℚ³ → ℚ

In [84]:
assert φ((1, 1, 2)) = 0
assert φ((1, 1, 0)) = 2

✓ φ((1, 1, 2)) = 0


✓ φ((1, 1, 0)) = 2


Homs are first-class, but a hom is not an object of the category it runs in: ascribing the map itself to `Mod(QQ)` is a morphism-is-not-an-object hold, refused by name.

In [85]:
let gm: QQ^3 -> QQ := (a, b, c) |-> a + b - c in Mod(QQ)

LeanError: a hom is a morphism, not an object of Mod(ℚ); ascribing it there is refused rather than read as membership — the arrow `(a, b, c) ↦ a + b - c ∈ ℚ³ → ℚ` already names its domain and codomain

A vector-valued hom is called, composed, and read by `ker` and `im` — the subobject presentations route to the same span machinery spans use.

In [86]:
let fh: ℚ³ → ℚ³ := (x, y, z) ↦ (2x + 3y + z, x - y, 3z - x)

fh := (x, y, z) ↦ (2x + 3y + z, x - y, -x + 3z) ∈ ℚ³ → ℚ³

In [87]:
assert fh((1, 0, 0)) = (2, 1, -1)

✓ fh((1, 0, 0)) = (2, 1, -1)


In [88]:
assert (fh ∘ fh)((1, 0, 0)) = (6, 1, -5)

✓ (fh ∘ fh)((1, 0, 0)) = (6, 1, -5)


In [89]:
assert ker fh = {0}

✓ ker fh = {0}


In [90]:
assert im fh = span_QQ{(1, 0, 0), (0, 1, 0), (0, 0, 1)}

✓ im fh = span_QQ{(1, 0, 0), (0, 1, 0), (0, 0, 1)}


## 10 · Elementary calculus

Limits are exact values where they exist — evaluated, not approximated.
A limit the backend cannot name is a refusal (§13), never a guess.

In [91]:
assert lim_{t → 0} sin(t)/t = 1
assert lim_{t → ∞} 1/t = 0

✓ lim_{t → 0} sin(t)/t = 1


✓ lim_{t → ∞} 1/t = 0


Definite integrals are exact:

In [92]:
assert ∫₀¹ t² dt = 1/3
assert ∫₀^π sin(t) dt = 2

✓ ∫₀¹ t² dt = 1/3


✓ ∫₀^π sin(t) dt = 2


Taylor expansion is a function into the ring of formal power series —
the jet of $f$ at a point. Truncation `ℝ[[t]]/(t^n)` is a quotient of
that ring, and the map is exact:

In [93]:
let expf := t ↦ exp(t) in ℝ → ℝ
let Tf := expf.taylor_expansion(0) in ℝ[[t]]
assert Tf ∈ ℝ[[t]]
map Tf to ℝ[[t]]/(t^6)

✓ Tf ∈ ℝ[[t]]


expf := t ↦ exp(t) ∈ ℝ → ℝ

Tf := 1 + t + (1/2)t^2 + (1/6)t^3 + (1/24)t^4 + (1/120)t^5 + (1/720)t^6 + (1/5040)t^7 + (1/40320)t^8 + (1/362880)t^9 + (1/3628800)t^10 + (1/39916800)t^11 + O(t^12) ∈ ℝ[[t]]

1 + t + (1/2)t^2 + (1/6)t^3 + (1/24)t^4 + (1/120)t^5 + O(t^6)

In [94]:
let g: ℝ → ℝ := t ↦ sin(t)
let Tg := g.taylor_expansion(0) in ℝ[[t]]
map Tg to ℝ[[t]]/(t^8)

g := t ↦ sin(t) ∈ ℝ → ℝ

Tg := t + (-1/6)t^3 + (1/120)t^5 + (-1/5040)t^7 + (1/362880)t^9 + (-1/39916800)t^11 + O(t^12) ∈ ℝ[[t]]

t + (-1/6)t^3 + (1/120)t^5 + (-1/5040)t^7 + O(t^8)

## 11 · A composed computation

Everything composes. The roots of $r$ over ℂ, their elementary
symmetric functions, and the companion matrix all answer from the same
polynomial:

In [95]:
let r(x) := x³ - 2x + 1 in ℚ[x]
let roots := {a ∈ ℂ | r(a) = 0} in 𝒫(ℂ)
assert |roots| = 3
assert 1 ∈ roots
assert ∑_{a ∈ roots} a = 0
assert ∏_{a ∈ roots} a = -1

algebraic numbers are presented through a fixed embedding ℚ̄ ↪ ℂ


✓ |roots| = 3


✓ 1 ∈ roots


✓ ∑_{a ∈ roots} a = 0


✓ ∏_{a ∈ roots} a = -1


r := x^3 - 2x + 1 ∈ ℚ[x]

roots := {-1/2 - (1/2)√5, -1/2 + (1/2)√5, 1}

In [96]:
let C := r.companion_matrix()
assert C.charpoly() = r
assert C.det() = -1
assert C.trace() = 0

✓ C.charpoly() = r


✓ C.det() = -1


✓ C.trace() = 0


C := [0, 0, -1; 1, 0, 2; 0, 1, 0] ∈ Mat₃(ℚ)

## 12 · Ellipses

`...` denotes an infinite sequence inferred from a finite pattern:

In [97]:
let X := {0, 1, 2, ...}
assert X = ℕ
let Y := {0, 2, 4, ...}
assert Y = 2ℕ
assert 8 ∈ Y
assert 9 ∉ Y

✓ X = ℕ


✓ Y = 2ℕ


✓ 8 ∈ Y


✓ 9 ∉ Y


X := {0, 1, ...}

Y := {0, 2, ...}

The alias layer: the backslash family and the ident aliases are accepted wherever the unicode form goes — `\NN`, `\in`, `\leq` are the same tokens as ℕ, ∈, ≤ — and SPEC's series binding uses the ASCII spellings throughout.

In [98]:
let Xa := {0, 1, 2, ...}

Xa := {0, 1, ...}

In [99]:
assert Xa = \NN

✓ Xa = \NN


In [100]:
assert 7/3 \in ℚ

✓ 7/3 ∈ ℚ


In [101]:
let fa: NN -> NN := n ↦ n^2

fa := n ↦ n^2 ∈ ℕ → ℕ

In [102]:
assert fa(3) = 9

✓ fa(3) = 9


In [103]:
let fs(t) = ∑_{n ∈ \NN} n^2 t^n \in ZZ[[t]]

fs := t + 4t^2 + 9t^3 + 16t^4 + ... ∈ ℤ[[t]]

In [104]:
assert [t^2]fs = 4

✓ [t^2]fs = 4


A guarded comprehension may be lazy: `primes` is the set of primes,
decided on membership — never enumerated.

In [105]:
let primes := {n in ℕ | n.is_prime()}
assert 13 ∈ primes
assert 15 ∉ primes

✓ 13 ∈ primes


✓ 15 ∉ primes


primes := {n ∈ ℕ | n.is_prime()}

Ellipses also fill a finite pattern: `CC[x_0, x_1, ..., x_9]` is the
polynomial ring in ten indeterminates.

In [106]:
let Rf := CC[x_0, x_1, ..., x_9]

LeanError: D[x_0, x_1, ..., x_n] — a polynomial algebra on a family of indeterminates (SPEC.md §Ellipses) — is not implemented: the spelling is reserved for it. One indeterminate — `ℂ[x]` — is available

A generating series is an element of the ring of formal power series —
coefficients extractable by `[t^k]`:

In [107]:
let sq(t) = ∑_{n ∈ ℕ} n^2 t^n ∈ ℤ[[t]]
assert [t^2]sq = 4
assert sq ∈ ℤ[[t]]/(t^5)
map sq to ℤ[[t]] / O(t^5)

LeanError: `E[[t]]/(t^5)` occurs here only as the target of `map … to` — a request that a series be presented by its first 5 coefficients. The quotient ring itself is not presented in CasDsl, so it answers no membership, cardinality or inclusion question

## 13 · The audit commands

The diagnostics are part of the language: `#capabilities` lists what each backend can do; `#capability_gaps` lists what is refused — `det` over ℤ/5, the ceiling below; `#canonical_maps` lists the registered identifications that drive `⊆`, `∈`, `map`, and coercion.

In [108]:
#capabilities

factor (x : R) [CommRing R] [IsDomain R] [UniqueFactorizationMonoid R]  ≐ UniqueFactorizationMonoid.factors
  element of ℤ → sage "factor_int"; element of ℚ[x] → sage "factor_poly_q"; element of ℤ[x] → sage "factor_poly_z"; element of ℂ[x] → sage "factor_poly_c"
  factor into irreducibles/primes with multiplicity
gcd (x : R) [CommRing R] [IsDomain R] [UniqueFactorizationMonoid R]  ≐ GCDMonoid.gcd
  element of ℤ → sage "gcd_int"
  a greatest common divisor, unique up to units
is_prime (x : R) [CommRing R] [IsDomain R] [UniqueFactorizationMonoid R]  ≐ Prime
  element of ℤ → sage "is_prime_int"
  primality: x is prime iff (x) is a nonzero prime ideal — so −7 is prime exactly as 7 is, (−7) = (7)
derivative (x : R) — declared on PolynomialElems  ≐ Polynomial.derivative
  element of _[x] → native "poly_derivative"
  the formal derivative d/dx, by exact coefficient arithmetic — computed natively
antiderivative (x : R) — declared on PolynomialElems  ≐ Polynomial.derivative
  element of ℤ[x] → 

In [109]:
#capability_gaps

  method            unimplemented for
  union             no route accepts: ℤ; ℚ; ℕ; ℤ[x]; x^3 + ℚ ⊆ ℚ[x]; {0,2,4,…}; {1,2,3} × {1,2,3}; 𝒫({1,2,3}); span_ℚ{(1,0,1), (0,1,1)} ≤ ℚ³; ℂ; ℤ[[t]]
  intersect         no route accepts: ℤ; ℚ; ℕ; ℤ[x]; x^3 + ℚ ⊆ ℚ[x]; {0,2,4,…}; {1,2,3} × {1,2,3}; 𝒫({1,2,3}); span_ℚ{(1,0,1), (0,1,1)} ≤ ℚ³; ℂ; ℤ[[t]]
  diff              no route accepts: ℤ; ℚ; ℕ; ℤ[x]; x^3 + ℚ ⊆ ℚ[x]; {0,2,4,…}; {1,2,3} × {1,2,3}; 𝒫({1,2,3}); span_ℚ{(1,0,1), (0,1,1)} ≤ ℚ³; ℂ; ℤ[[t]]
  symdiff           no route accepts: ℤ; ℚ; ℕ; ℤ[x]; x^3 + ℚ ⊆ ℚ[x]; {0,2,4,…}; {1,2,3} × {1,2,3}; 𝒫({1,2,3}); span_ℚ{(1,0,1), (0,1,1)} ≤ ℚ³; ℂ; ℤ[[t]]
  gcd               no route accepts: x^3 − 2x + 1 ∈ ℤ[x]; sample q ∈ ℚ[x]
  is_prime          no route accepts: x^3 − 2x + 1 ∈ ℤ[x]; sample q ∈ ℚ[x]
  companion_matrix  no route accepts: x^3 − 2x + 1 ∈ ℤ[x]; x + 1 ∈ ℤ/5[x]
  det               no route accepts: [1,2;3,4] ∈ Mat₂(ℤ/5)
  inverse           no route accepts: [1,2;3,4] ∈ Mat₂(ℤ/5)
  rank    

In [110]:
#canonical_maps

  map         op
  ℕ → ℤ       identity
              ℕ ⊆ ℤ: an element of ℕ already IS an integer (they share the `Value.int` representation), so the injection moves no data
  ℕ → ℚ       intToRat
              ℕ ⊆ ℚ: the composite of ℕ ⊆ ℤ ⊆ ℚ, registered explicitly because the coercion layer takes ONE hop (it does not compose rules)
  ℤ → ℚ       intToRat
              ℤ ⊆ ℚ: the fraction field of ℤ — the notebook's `map p to ℚ[x]` is this rule applied coefficient-wise
  ℤ → ℤ/_     intToMod
              ℤ → ℤ/n for EVERY modulus n (one rule, by `anyMod`): the ring quotient — an integer naming its residue class, which is what an ascription such as `let x := 7 in ℤ/5` inserts
  ℕ → ℝ       identity
              ℕ ⊆ ℝ: a natural number is a real number
  ℤ → ℝ       identity
              ℤ ⊆ ℝ: an integer is a real number
  ℚ → ℝ       identity
              ℚ ⊆ ℝ: SPEC.md's own chain link — every rational IS a real, and the value presenting it does not change
  ℕ → ℂ       identit

## 14 · Documented ceiling

Not every mathematically meaningful operation has an implementation yet.
Ask for one and the system refuses, naming the operation, the object it
was asked on, and why no backend can answer — an explicit gap, not a
crash and not a hidden method.

Here `det` is known on Mat₂(ℤ/5) — the category declares the method —
but no backend provides it for matrices over ℤ/5 yet. Over ℚ it works
(we just used it); over ℤ/5 it is a gap.


In [111]:
let N := [1, 2; 3, 4] in Mat₂(ℤ/5)

N := [1, 2; 3, 4] ∈ Mat₂(ℤ/5)

In [112]:
N.det()

LeanError: NoImplementation: 'det' is mathematically available for x = [1, 2; 3, 4] ∈ Mat₂(ℤ/5) (declared directly on MatrixElems(2, ℤ/5)), but the registered routes — element of Mat(_, ℚ) → sage "mat_det_q" — accept none of it.
The method stays available on MatrixElems(2, ℤ/5); the mathematics is not narrowed — an implementation for this presentation does not exist yet.